In [5]:
import re
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display, Markdown

In [6]:
def extract_letter(prediction: str) -> str | None:
    """
    Extract A/B/C/D from a model prediction.

    Patterns are tried in priority order so that the most specific and
    unambiguous matches win before falling back to the first standalone letter.

    Returns the uppercase letter or None if no match is found.
    """
    if not prediction or not prediction.strip():
        return None

    text = prediction.strip()

    patterns = [
        # Bare letter at the start of the string (e.g. "A", "A.", "A)")
        r"^([A-D])[.)\s,]?",
        # "answer is/was/= A", "answer: A"
        r"(?:answer\s*(?:is|was|[:=])\s*)([A-D])\b",
        # Parenthesised letter: "(A)", "[A]"
        r"[\(\[（]([A-D])[\)\]）]",
        # "The correct answer is A", "correct option is A"
        r"correct\s+(?:answer|option|choice)\s+(?:is\s+)?([A-D])\b",
        # "option A", "choice A"
        r"(?:option|choice)\s+([A-D])\b",
        # "A)" or "A." anywhere in the string
        r"\b([A-D])[.)]",
        # Last resort: any standalone A-D
        r"\b([A-D])\b",
    ]

    for pattern in patterns:
        m = re.search(pattern, text, re.IGNORECASE)
        if m:
            return m.group(1).upper()

    return None


def load_results(path: str) -> pd.DataFrame:
    df = pd.read_json(path, lines=True)
    df["extracted"] = df["prediction"].apply(extract_letter)
    df["correct"] = df["extracted"] == df["answer"]
    return df


def compute_accuracy(path: str) -> dict:
    df = load_results(path)
    n_total = len(df)
    n_correct = df["correct"].sum()
    n_unparsed = df["extracted"].isna().sum()
    return {
        "accuracy": n_correct / n_total,
        "n_correct": int(n_correct),
        "n_total": n_total,
        "n_unparsed": int(n_unparsed),
    }


def compute_accuracy_by_category(path: str) -> pd.DataFrame:
    df = load_results(path)
    rows = []
    for (cat, l2), grp in df.groupby(["category", "l2_category"]):
        rows.append({
            "category": cat,
            "l2_category": l2,
            "accuracy": grp["correct"].mean(),
            "n": len(grp),
        })
    return pd.DataFrame(rows).sort_values(["category", "l2_category"]).reset_index(drop=True)


def results_table(result_files: dict) -> pd.DataFrame:
    """
    Build a summary table for a set of result files.

    Args:
        result_files: dict mapping display name -> path to .jsonl result file.

    Returns a DataFrame with one row per run.
    """
    rows = []
    for name, path in result_files.items():
        p = Path(path)
        if not p.exists():
            rows.append({"name": name, "accuracy": float("nan"), "n_correct": None,
                         "n_total": None, "n_unparsed": None, "path": path})
            continue
        stats = compute_accuracy(path)
        rows.append({"name": name, **stats, "path": path})
    df = pd.DataFrame(rows).set_index("name")
    df["accuracy_%"] = (df["accuracy"] * 100).round(2)
    return df


def show_sample(result_files: dict, ref_index: int = 0, base_key: str = None):
    """
    Display one example across multiple result files to compare predictions.
    """
    if base_key is None:
        base_key = next(iter(result_files))

    base_df = load_results(result_files[base_key])
    row = base_df.iloc[ref_index]
    display(Markdown(
        f"**Sample index:** {ref_index}  \n"
        f"**Question:** {row['question']}  \n"
        f"**Ground Truth:** **{row['answer']}**"
    ))

    for name, path in result_files.items():
        df = load_results(path)
        r = df.iloc[ref_index]
        extracted = r["extracted"] or "(unparsed)"
        correct_marker = "✓" if r["correct"] else "✗"
        display(Markdown(
            f"---\n### {name} — extracted: **{extracted}** {correct_marker}\n\n"
            f"Raw prediction: `{r['prediction'].strip()}`"
        ))

---
# google/gemma-3-4b-pt

In [19]:
df = pd.read_json("results/mmstar/gemma-3-4b-pt/gemma-3-4b-pt-base.jsonl", lines=True)
df["answer"].value_counts()

answer
B    447
A    429
D    315
C    309
Name: count, dtype: int64

In [8]:
result_files = {
    "base":     "results/mmstar/gemma-3-4b-pt/gemma-3-4b-pt-base.jsonl",
    "sft-20k":  "results/mmstar/gemma-3-4b-pt/gemma-3-4b-pt-sft-20000.jsonl",
    "dpo-1250": "results/mmstar/gemma-3-4b-pt/gemma-3-4b-pt-dpo-1250.jsonl",
}
results_table(result_files)

,accuracy,n_correct,n_total,n_unparsed,path,accuracy_%
name,,,,,,
base,0.006000,9,1500,1460,results/mmstar/gemma-3-4b-pt/gemma-3-4b-pt-bas...,0.60
sft-20k,0.064667,97,1500,1130,results/mmstar/gemma-3-4b-pt/gemma-3-4b-pt-sft...,6.47
dpo-1250,0.064667,97,1500,1078,results/mmstar/gemma-3-4b-pt/gemma-3-4b-pt-dpo...,6.47


---
# meta-llama/Llama-3.2-11B-Vision

In [9]:
result_files = {
    "base":     "results/mmstar/Llama-3.2-11B-Vision/Llama-3.2-11B-Vision-base.jsonl",
    "sft-20k":  "results/mmstar/Llama-3.2-11B-Vision/Llama-3.2-11B-Vision-sft-20000.jsonl",
    "dpo-1250": "results/mmstar/Llama-3.2-11B-Vision/Llama-3.2-11B-Vision-dpo-1250.jsonl",
}
results_table(result_files)

,accuracy,n_correct,n_total,n_unparsed,path,accuracy_%
name,,,,,,
base,0.361333,542,1500,3,results/mmstar/Llama-3.2-11B-Vision/Llama-3.2-...,36.13
sft-20k,0.290667,436,1500,203,results/mmstar/Llama-3.2-11B-Vision/Llama-3.2-...,29.07
dpo-1250,0.296667,445,1500,74,results/mmstar/Llama-3.2-11B-Vision/Llama-3.2-...,29.67


---
# Qwen/Qwen3.5-4B-Base

In [10]:
result_files = {
    "base":     "results/mmstar/Qwen3.5-4B-Base/Qwen3.5-4B-Base-base.jsonl",
    "sft-20k":  "results/mmstar/Qwen3.5-4B-Base/Qwen3.5-4B-Base-sft-20000.jsonl",
    "dpo-1250": "results/mmstar/Qwen3.5-4B-Base/Qwen3.5-4B-Base-dpo-1250.jsonl",
}
results_table(result_files)

,accuracy,n_correct,n_total,n_unparsed,path,accuracy_%
name,,,,,,
base,0.024000,36,1500,1310,results/mmstar/Qwen3.5-4B-Base/Qwen3.5-4B-Base...,2.40
sft-20k,0.031333,47,1500,1262,results/mmstar/Qwen3.5-4B-Base/Qwen3.5-4B-Base...,3.13
dpo-1250,0.030667,46,1500,1292,results/mmstar/Qwen3.5-4B-Base/Qwen3.5-4B-Base...,3.07


---
# Inspect individual samples

In [20]:
result_files = {
    "base":     "results/mmstar/Llama-3.2-11B-Vision/Llama-3.2-11B-Vision-base.jsonl",
    "sft-20k":  "results/mmstar/Llama-3.2-11B-Vision/Llama-3.2-11B-Vision-sft-20000.jsonl",
    "dpo-1250": "results/mmstar/Llama-3.2-11B-Vision/Llama-3.2-11B-Vision-dpo-1250.jsonl",
}
show_sample(result_files, ref_index=0)

**Sample index:** 0  
**Question:** Which option describe the object relationship in the image correctly?
Options: A: The suitcase is on the book., B: The suitcase is beneath the cat., C: The suitcase is beneath the bed., D: The suitcase is beneath the book.  
**Ground Truth:** **A**

---
### base — extracted: **B** ✗

Raw prediction: `B. The suitcase in the picture is on top of the book. <OCR`

---
### sft-20k — extracted: **B** ✗

Raw prediction: `B`

---
### dpo-1250 — extracted: **A** ✓

Raw prediction: `A`